# Tutorial: TL-008 Custom Text Extraction Demo

Audience:
- Engineers validating the TL-008 extraction flow on ad hoc text.

Prerequisites:
- Run this notebook from the repo root environment after `uv sync`.
- LM Studio's OpenAI-compatible endpoint must already be running at the repo-configured extraction base URL.
- The local embedding runtime dependencies used by the repo must be available on this machine. In practice that means the `sentence-transformers` stack from the repo environment and access to the configured embedding model.

Learning goals:
- Paste custom text into one cell and inspect the exact chunk text sent into TL-008 extraction.
- See the real extraction prompt, the model's validated `summary` and string `tags` payload, and whether JSON came from visible content or the validated reasoning-content compatibility fallback.
- Confirm that `models.extraction` uses Qwen2.5 while the general LLM config can remain separate.
- Keep the final TL-008-style payload visible so the end-to-end chunk, embedding, and aggregation shape stays easy to inspect.


## Outline

1. Edit the input text that will be chunked and sent to the live LM Studio-backed extractor.
2. Validate the repo config and fail fast if the LM Studio endpoint or required local dependencies are unavailable.
3. Inspect the strict TL-008 `summary` plus `tags: list[str]` extraction schema.
4. Build repo-native `ParsedDocument`, `DocumentRecord`, and `ChunkRecord` objects.
5. Run live chunk extraction with recording enabled so you can inspect chunk text, prompt text, structured output, and response metadata.
6. Attach embeddings, aggregate document metadata, and inspect the final TL-008 payload.


In [16]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import UTC, datetime
import hashlib
import importlib
import json
from pathlib import Path
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse
from urllib.request import Request, urlopen

REPO_ROOT = Path.cwd().resolve()
print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/pelmeshek1706/Desktop/projects/knowledge_agent/output/jupyter-notebook


In [17]:
from personal_kb.core.config_loader import load_config
config = load_config('/Users/pelmeshek1706/Desktop/projects/knowledge_agent/configs/default.yaml')

print("resolved general llm base_url:", config.models.llm.base_url)
print("resolved general llm model:", config.models.llm.model_name)
print("resolved extraction base_url:", config.models.extraction.base_url)
print("resolved extraction model:", config.models.extraction.model_name)
print("resolved embedding model:", config.models.embedding.model_name)


resolved general llm base_url: http://localhost:1234/v1
resolved general llm model: qwen2.5-1.5b-instruct
resolved extraction base_url: http://localhost:1234/v1
resolved extraction model: qwen2.5-1.5b-instruct
resolved embedding model: Qwen/Qwen3-Embedding-0.6B


## Step 1 - Paste or edit custom text

Update `user_text` and rerun the notebook from this cell downward.

This notebook is intentionally live-only. If LM Studio or the repo's local embedding runtime is unavailable, the setup cell will stop with a clear error instead of switching to fake data.


In [18]:
user_text = """Personal KB is a local-first GraphRAG knowledge base built in Python.
It uses LM Studio to run Qwen models through an OpenAI-compatible API,
LangGraph for agent orchestration, Neo4j for graph storage, and strict structured JSON
for chunk metadata extraction. The TL-008 extraction path should produce useful tags
for systems, tools, models, libraries, technologies, documents, concepts, and domain terms."""

document_title = "TL-008 Qwen2.5 extraction demo"
chunk_size = 420
chunk_overlap = 40


## Step 2 - Import the current TL-008 building blocks

The notebook uses the same chunker, prompt builder, extractor, extraction service, embedding service, and aggregation service that the repo uses for TL-008. The only notebook-specific helper is a small recording wrapper so each live model call stays inspectable.


In [19]:
from personal_kb.chunking.txt_chunker import TxtChunker
from personal_kb.core import load_config
from personal_kb.core.errors import (
    EmbeddingError,
    ExtractionError,
    LLMError,
    ModelProviderUnavailableError,
    StructuredOutputError,
)
from personal_kb.extraction.prompts import (
    STRICT_JSON_SYSTEM_PROMPT,
    build_chunk_extraction_prompt,
    build_document_aggregation_prompt,
)
from personal_kb.extraction.structured_extractor import ChunkExtractionPayload, StructuredExtractor
from personal_kb.ingestion.document_aggregation_service import DocumentAggregationService
from personal_kb.ingestion.embedding_service import EmbeddingService
from personal_kb.ingestion.extraction_service import ExtractionService
from personal_kb.models.embedding_client import EmbeddingClient
from personal_kb.models.llm_client import LLMClient
from personal_kb.models.structured_extraction_client import (
    StructuredExtractionClient,
    StructuredExtractionResult,
)
from personal_kb.schemas.common import SourceRef
from personal_kb.schemas.document import (
    DocumentMetadata,
    DocumentRecord,
    ParsedDocument,
    RawDocumentHashes,
)


In [20]:
print(STRICT_JSON_SYSTEM_PROMPT)
print()
print(json.dumps(ChunkExtractionPayload.model_json_schema(), indent=2))


You are a precise information extraction system. Return only valid JSON that matches the provided schema exactly. Do not include markdown, comments, explanations, or extra keys.

{
  "additionalProperties": false,
  "properties": {
    "summary": {
      "title": "Summary",
      "type": "string"
    },
    "tags": {
      "items": {
        "type": "string"
      },
      "maxItems": 10,
      "title": "Tags",
      "type": "array"
    }
  },
  "required": [
    "summary"
  ],
  "title": "ChunkExtractionPayload",
  "type": "object"
}


## Step 3 - Validate the live runtime and build services

This cell is the gatekeeper for the notebook. It verifies the repo config, confirms that the LM Studio OpenAI-compatible endpoint is reachable for `models.extraction`, checks the required local Python dependencies, and then constructs the live TL-008 services.

If this cell raises an error, fix the environment first. The notebook does not provide a stub fallback.


In [21]:
def digest_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def build_parsed_document(text: str, title: str) -> ParsedDocument:
    timestamp = datetime.now(UTC)
    source_id = "demo://custom-text"
    return ParsedDocument(
        source_id=source_id,
        file_path=source_id,
        file_name="custom_text_demo.txt",
        file_extension="txt",
        title=title,
        raw_text=text.strip(),
        metadata=DocumentMetadata(created_at=timestamp, modified_at=timestamp),
        hashes=RawDocumentHashes(
            raw_bytes_hash=digest_text(text),
            extracted_text_hash=digest_text(text),
        ),
        structured_blocks=[
            {
                "kind": "paragraph",
                "char_count": len(text.strip()),
            }
        ],
    )


def build_document_record(parsed_document: ParsedDocument) -> DocumentRecord:
    content_hash = digest_text(parsed_document.raw_text)
    return DocumentRecord(
        document_id=digest_text(f"{parsed_document.source_id}:{content_hash}")[:16],
        source_id=parsed_document.source_id,
        file_path=parsed_document.file_path,
        file_name=parsed_document.file_name,
        file_extension=parsed_document.file_extension,
        document_type="text",
        title=parsed_document.title,
        ingested_at=datetime.now(UTC),
        raw_bytes_hash=parsed_document.hashes.raw_bytes_hash or content_hash,
        extracted_text_hash=parsed_document.hashes.extracted_text_hash,
        content_hash=content_hash,
    )


@dataclass
class RecordedExtractionCall:
    stage: str
    prompt: str
    system_prompt: str | None
    response_schema: str
    thinking_mode: str | None
    temperature: float
    max_tokens: int | None
    attempts: int
    validator_notes: list[str]
    response_content: str
    reasoning_content: str | None
    metadata: dict[str, Any]
    parsed_value: dict[str, Any]


class RecordingStructuredExtractionClient:
    def __init__(self, delegate: StructuredExtractionClient) -> None:
        self._delegate = delegate
        self.history: list[RecordedExtractionCall] = []

    def extract(
        self,
        prompt: str,
        *,
        response_schema: type[Any],
        system_prompt: str | None = None,
        validator: Any = None,
        thinking_mode: str | None = None,
        temperature: float = 0.0,
        max_tokens: int | None = None,
        max_retries: int | None = None,
    ) -> StructuredExtractionResult[Any]:
        result = self._delegate.extract(
            prompt,
            response_schema=response_schema,
            system_prompt=system_prompt,
            validator=validator,
            thinking_mode=thinking_mode,
            temperature=temperature,
            max_tokens=max_tokens,
            max_retries=max_retries,
        )
        schema_name = response_schema.__name__
        stage = "document_aggregation" if schema_name == "DocumentSummaryPayload" else "chunk_extraction"
        self.history.append(
            RecordedExtractionCall(
                stage=stage,
                prompt=prompt,
                system_prompt=system_prompt,
                response_schema=schema_name,
                thinking_mode=thinking_mode,
                temperature=temperature,
                max_tokens=max_tokens,
                attempts=result.attempts,
                validator_notes=list(result.validator_notes),
                response_content=result.response.content,
                reasoning_content=result.response.reasoning_content,
                metadata=result.response.metadata.model_dump(mode="json"),
                parsed_value=result.value.model_dump(mode="json"),
            )
        )
        return result


def find_structured_output_error(exc: BaseException) -> StructuredOutputError | None:
    current: BaseException | None = exc
    seen: set[int] = set()
    while current is not None and id(current) not in seen:
        if isinstance(current, StructuredOutputError):
            return current
        seen.add(id(current))
        current = current.__cause__ or current.__context__
    return None


def print_structured_output_failure(
    *,
    stage: str,
    prompt: str | None,
    exc: BaseException,
    chunk_text: str | None = None,
) -> None:
    print("=" * 100)
    print(f"{stage} failed")
    print("-" * 100)
    print("Failure reason")
    print(str(exc))
    if chunk_text is not None:
        print()
        print("Chunk text sent to the model")
        print(chunk_text)
    if prompt is not None:
        print()
        print("Prompt sent to the model")
        print(prompt)

    structured_error = find_structured_output_error(exc)
    if structured_error is None:
        return

    response_metadata = getattr(structured_error, "response_metadata", None)
    if response_metadata is not None:
        print()
        print("Provider metadata")
        print(json.dumps(response_metadata.model_dump(mode="json"), indent=2))

    raw_provider_response = getattr(structured_error, "raw_provider_response", None)
    if raw_provider_response:
        print()
        print("Raw provider response")
        print(json.dumps(raw_provider_response, indent=2))


def require_python_dependency(module_name: str, install_hint: str) -> None:
    try:
        importlib.import_module(module_name)
    except ImportError as exc:
        raise RuntimeError(
            f"Missing required dependency '{module_name}'. {install_hint}"
        ) from exc


def require_lm_studio_endpoint(base_url: str, timeout_seconds: float) -> dict[str, Any]:
    parsed = urlparse(base_url)
    if not parsed.scheme or not parsed.netloc:
        raise RuntimeError(
            f"Configured LM Studio base URL is invalid: {base_url!r}"
        )

    models_url = base_url.rstrip("/") + "/models"
    request = Request(models_url, headers={"Authorization": "Bearer lm-studio"})
    try:
        with urlopen(request, timeout=timeout_seconds) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            "LM Studio endpoint responded with an HTTP error while checking "
            f"{models_url}: {exc.code} {exc.reason}. Details: {detail[:300]}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(
            "LM Studio endpoint is not reachable at "
            f"{models_url}. Start the OpenAI-compatible server in LM Studio first. "
            f"Original error: {exc.reason}"
        ) from exc

    if not isinstance(payload, dict):
        raise RuntimeError(
            f"LM Studio health check returned a non-object payload from {models_url}."
        )
    return payload


def build_live_services() -> tuple[
    ExtractionService,
    EmbeddingService,
    DocumentAggregationService,
    RecordingStructuredExtractionClient,
    dict[str, Any],
]:
    import os
    os.environ.pop("LMSTUDIO_MODEL", None)
    os.environ.pop("LMSTUDIO_URL", None)
    os.environ.pop("LMSTUDIO_EMBED_MODEL", None)
    config = load_config("/Users/pelmeshek1706/Desktop/projects/knowledge_agent/configs/default.yaml")
    print("USED_GENERAL_LLM_MODEL:", config.models.llm.model_name)
    print("USED_EXTRACTION_MODEL:", config.models.extraction.model_name)
    require_python_dependency("openai", "Run `uv sync` in the repo environment.")
    require_python_dependency(
        "sentence_transformers",
        "Run `uv sync` and make sure the local embedding runtime dependencies are installed.",
    )
    models_payload = require_lm_studio_endpoint(
        config.models.extraction.base_url,
        config.models.extraction.timeout_seconds,
    )

    llm_client = LLMClient(config.models.extraction)
    recording_client = RecordingStructuredExtractionClient(
        StructuredExtractionClient(llm_client)
    )
    extractor = StructuredExtractor(recording_client)
    embedding_client = EmbeddingClient(config.models.embedding)

    runtime_summary = {
        "mode": "live_lm_studio_required",
        "llm": {
            "provider": config.models.llm.provider,
            "base_url": config.models.llm.base_url,
            "model_name": config.models.llm.model_name,
            "default_thinking_mode": config.models.llm.default_thinking_mode,
            "structured_output_retries": config.models.llm.structured_output_retries,
        },
        "extraction": {
            "provider": config.models.extraction.provider,
            "base_url": config.models.extraction.base_url,
            "model_name": config.models.extraction.model_name,
            "default_thinking_mode": config.models.extraction.default_thinking_mode,
            "structured_output_retries": config.models.extraction.structured_output_retries,
        },
        "embedding": {
            "provider": config.models.embedding.provider,
            "model_name": config.models.embedding.model_name,
            "dimension": config.models.embedding.dimension,
            "instruction_aware": config.models.embedding.instruction_aware,
            "normalize_embeddings": config.models.embedding.normalize_embeddings,
        },
        "lm_studio_models_response_keys": sorted(models_payload.keys()),
        "lm_studio_model_count": len(models_payload.get("data", []))
        if isinstance(models_payload.get("data"), list)
        else None,
    }

    return (
        ExtractionService(extractor),
        EmbeddingService(embedding_client),
        DocumentAggregationService(extractor),
        recording_client,
        runtime_summary,
    )


def build_final_payload(document: DocumentRecord, chunks: list[Any]) -> dict[str, Any]:
    return {
        "document": document.model_dump(mode="json"),
        "chunks": [
            {
                **chunk.model_dump(mode="json"),
                "embedding": [round(value, 4) for value in chunk.embedding],
            }
            for chunk in chunks
        ],
    }


extraction_service, embedding_service, aggregation_service, recording_client, runtime_info = (
    build_live_services()
)

print(json.dumps(runtime_info, indent=2))


USED_GENERAL_LLM_MODEL: qwen2.5-1.5b-instruct
USED_EXTRACTION_MODEL: qwen2.5-1.5b-instruct
{
  "mode": "live_lm_studio_required",
  "llm": {
    "provider": "lmstudio_openai_compatible",
    "base_url": "http://localhost:1234/v1",
    "model_name": "qwen2.5-1.5b-instruct",
    "default_thinking_mode": "non_thinking",
    "structured_output_retries": 2
  },
  "extraction": {
    "provider": "lmstudio_openai_compatible",
    "base_url": "http://localhost:1234/v1",
    "model_name": "qwen2.5-1.5b-instruct",
    "default_thinking_mode": "non_thinking",
    "structured_output_retries": 2
  },
  "embedding": {
    "provider": "local",
    "model_name": "Qwen/Qwen3-Embedding-0.6B",
    "dimension": 1024,
    "instruction_aware": true,
    "normalize_embeddings": true
  },
  "lm_studio_models_response_keys": [
    "data",
    "object"
  ],
  "lm_studio_model_count": 2
}


## Step 4 - Create the document record and chunk setup

This uses the repo's `TxtChunker`, so chunk IDs, `source_ref`, and overlap behavior follow the same path as production ingestion for plain text. The output below keeps both per-chunk previews and the full text available for inspection.


In [22]:
parsed_document = build_parsed_document(user_text, document_title)
document_record = build_document_record(parsed_document)
chunker = TxtChunker(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
chunks = chunker.chunk(parsed_document, document_id=document_record.document_id)

print("Document record")
print(json.dumps(document_record.model_dump(mode="json"), indent=2, default=str))
print()
print(f"Chunk count: {len(chunks)}")
print(json.dumps(
    [
        {
            "chunk_id": chunk.chunk_id,
            "chunk_index": chunk.chunk_index,
            "char_count": chunk.char_count,
            "source_ref": chunk.source_ref.model_dump(mode="json"),
            "text": chunk.text,
        }
        for chunk in chunks
    ],
    indent=2,
))


Document record
{
  "document_id": "6e53de2be185f83a",
  "source_id": "demo://custom-text",
  "source_type": "local_file",
  "file_path": "demo://custom-text",
  "file_name": "custom_text_demo.txt",
  "file_extension": "txt",
  "document_type": "text",
  "title": "TL-008 Qwen2.5 extraction demo",
  "normalized_title": "tl-008 qwen2.5 extraction demo",
  "summary": null,
  "tags": [],
  "entities": [],
  "created_at": null,
  "modified_at": null,
  "ingested_at": "2026-05-14T09:02:27.667737Z",
  "raw_bytes_hash": "91bfe09d8460d1cae81049cc5993e734941dfd018c0b203e49e7d48aba7ec900",
  "extracted_text_hash": "91bfe09d8460d1cae81049cc5993e734941dfd018c0b203e49e7d48aba7ec900",
  "content_hash": "91bfe09d8460d1cae81049cc5993e734941dfd018c0b203e49e7d48aba7ec900",
  "is_duplicate": false,
  "canonical_document_id": null
}

Chunk count: 1
[
  {
    "chunk_id": "fed6995f-b4b5-5251-b59c-9b2dbfb22a30",
    "chunk_index": 0,
    "char_count": 404,
    "source_ref": {
      "file_path": "demo://custom

## Step 5 - Run live chunk-level structured extraction

The next cell runs the real TL-008 chunk extraction path against LM Studio with `config.models.extraction`. For each chunk it shows:
- the exact chunk text;
- the exact extraction prompt built by `build_chunk_extraction_prompt`;
- the model's validated `summary`, string `tags`, and typed `entities` payload;
- the resulting normalized `TagRecord` and `EntityRecord` objects on the chunk;
- response metadata, including `structured_output_source`, warnings, and any reasoning content surfaced by the provider.


In [ ]:
recording_client.history.clear()
try:
    enriched_chunks = extraction_service.enrich_chunks(chunks)
except ExtractionError as exc:
    failed_chunk = chunks[0] if chunks else None
    print_structured_output_failure(
        stage="Chunk extraction",
        prompt=(build_chunk_extraction_prompt(failed_chunk) if failed_chunk else None),
        exc=exc,
        chunk_text=(failed_chunk.text if failed_chunk else None),
    )
    raise

chunk_calls = [call for call in recording_client.history if call.stage == "chunk_extraction"]
t = []
for chunk, call in zip(enriched_chunks, chunk_calls, strict=True):
    print("=" * 100)
    print(f"Chunk {chunk.chunk_id} (index={chunk.chunk_index})")
    print("-" * 100)
    print("Chunk text sent to the model")
    print(chunk.text)
    print()
    print("Extraction prompt")
    print(call.prompt)
    print()
    print("Structured output returned by the model")
    print(json.dumps(call.parsed_value, indent=2))
    print()
    print("LLM response metadata")
    print(json.dumps(call.metadata, indent=2))
    print()
    print("Structured output source")
    print(call.metadata.get("structured_output_source") or "unknown")
    if call.validator_notes:
        print()
        print("Validator notes")
        print(json.dumps(call.validator_notes, indent=2))
    if call.reasoning_content:
        print()
        print("Reasoning content returned by the provider")
        print(call.reasoning_content)
    print()

print("Chunk extraction summary")

print(json.dumps(
    [
        {
            "chunk_id": chunk.chunk_id,
            "summary": chunk.summary,
            "tags": [tag.model_dump(mode="json") for tag in chunk.tags],
            "entities": [entity.model_dump(mode="json") for entity in chunk.entities],
        }
        for chunk in enriched_chunks
    ],
    indent=2,
))


Chunk fed6995f-b4b5-5251-b59c-9b2dbfb22a30 (index=0)
----------------------------------------------------------------------------------------------------
Chunk text sent to the model
Personal KB is a local-first GraphRAG knowledge base built in Python.
It uses LM Studio to run Qwen models through an OpenAI-compatible API,
LangGraph for agent orchestration, Neo4j for graph storage, and strict structured JSON
for chunk metadata extraction. The TL-008 extraction path should produce useful tags
for systems, tools, models, libraries, technologies, documents, concepts, and domain terms.

Extraction prompt
Task:
Extract retrieval-ready metadata from the chunk below.

Rules:
- Use only facts grounded in the provided chunk text.
- Write `summary` as 1 to 3 sentences and keep it concise.
- Return `tags` as 4 to 10 short strings useful for retrieval.
- Include important names, organizations, projects, systems, tools, models,
  libraries, technologies, documents, concepts, and domain terms as tags

## Step 6 - Attach embeddings

The embedding service now runs over the enriched chunks using the repo-configured local embedding model. This remains live as well, so failures here usually mean the local embedding runtime or model setup is incomplete.


In [24]:
try:
    embedded_chunks = embedding_service.embed_chunks(enriched_chunks)
except (EmbeddingError, ModelProviderUnavailableError) as exc:
    raise RuntimeError(
        "Live embeddings failed. Confirm that the repo's local embedding dependencies "
        "and configured model are available before rerunning this notebook."
    ) from exc

print(json.dumps(
    [
        {
            "chunk_id": chunk.chunk_id,
            "embedding_model": chunk.embedding_model,
            "embedding_dimension": chunk.embedding_dimension,
            "embedding_preview": [round(value, 4) for value in chunk.embedding[:12]],
            "full_vector_length": len(chunk.embedding),
        }
        for chunk in embedded_chunks
    ],
    indent=2,
))


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 10362.41it/s]


[
  {
    "chunk_id": "fed6995f-b4b5-5251-b59c-9b2dbfb22a30",
    "embedding_model": "Qwen/Qwen3-Embedding-0.6B",
    "embedding_dimension": 1024,
    "embedding_preview": [
      0.0074,
      0.0243,
      -0.0101,
      -0.1064,
      -0.014,
      -0.0145,
      -0.0093,
      -0.0069,
      -0.0195,
      0.0371,
      -0.0192,
      -0.0608
    ],
    "full_vector_length": 1024
  }
]


## Step 7 - Aggregate document-level metadata

The aggregation service asks the extractor for one live document summary using the chunk-level metadata. The prompt preview and metadata below make that final LLM step inspectable too.


In [25]:
try:
    aggregated_document = aggregation_service.aggregate_document(document_record, embedded_chunks)
except ExtractionError as exc:
    print_structured_output_failure(
        stage="Document aggregation",
        prompt=build_document_aggregation_prompt(document_record, embedded_chunks),
        exc=exc,
    )
    raise

document_call = next(
    call for call in reversed(recording_client.history)
    if call.stage == "document_aggregation"
)

print("Document aggregation prompt")
print(document_call.prompt)
print()
print("Structured output returned by the model")
print(json.dumps(document_call.parsed_value, indent=2))
print()
print("LLM response metadata")
print(json.dumps(document_call.metadata, indent=2))
print()
print("Structured output source")
print(document_call.metadata.get("structured_output_source") or "unknown")
if document_call.validator_notes:
    print()
    print("Validator notes")
    print(json.dumps(document_call.validator_notes, indent=2))
if document_call.reasoning_content:
    print()
    print("Reasoning content returned by the provider")
    print(document_call.reasoning_content)
print()
print("Aggregated document metadata")
print(json.dumps(aggregated_document.model_dump(mode="json"), indent=2))


Document aggregation prompt
Task:
Create one document-level summary from chunk-level metadata.

Rules:
- Use only the chunk summaries and metadata provided below.
- Write `summary` as 2 to 4 sentences and keep it concise.
- Emphasize the document's main subjects, actors, and outcomes.
- Do not invent facts that are not supported by the chunk metadata.
- Return only valid JSON that satisfies the schema.

Document metadata:
- document_id: 6e53de2be185f83a
- title: TL-008 Qwen2.5 extraction demo
- file_name: custom_text_demo.txt
- document_type: text

Chunk metadata:
- chunk_id: fed6995f-b4b5-5251-b59c-9b2dbfb22a30
  chunk_index: 0
  summary: Personal KB is a local-first GraphRAG knowledge base built in Python using Qwen models through an OpenAI-compatible API, LangGraph for agent orchestration, Neo4j for graph storage, and strict structured JSON for metadata extraction.
  tags: Qwen, OpenAI, Neo4j, LangGraph, Personal KB, GraphRAG, Python, LM Studio
  entities: none

Structured output re

## Step 8 - Final TL-008-style payload

This final object keeps the same shape as the TL-008 example: one document plus enriched chunks, each with summary, tags, entities, and embeddings.


In [26]:
final_payload = build_final_payload(aggregated_document, embedded_chunks)
print(json.dumps(final_payload, indent=2))


{
  "document": {
    "document_id": "6e53de2be185f83a",
    "source_id": "demo://custom-text",
    "source_type": "local_file",
    "file_path": "demo://custom-text",
    "file_name": "custom_text_demo.txt",
    "file_extension": "txt",
    "document_type": "text",
    "title": "TL-008 Qwen2.5 extraction demo",
    "normalized_title": "tl-008 qwen2.5 extraction demo",
    "summary": "The document describes the creation of a local-first GraphRAG knowledge base named Personal KB using Python and Qwen models through an OpenAI-compatible API. It also mentions the use of LangGraph for agent orchestration, Neo4j for graph storage, and strict structured JSON for metadata extraction.",
    "tags": [
      {
        "tag_id": "tag::qwen",
        "name": "Qwen",
        "normalized_name": "qwen",
        "confidence": null,
        "source": "llm_extraction",
        "source_chunks": [
          "fed6995f-b4b5-5251-b59c-9b2dbfb22a30"
        ],
        "created_at": null,
        "updated_at":

## Expected output and troubleshooting

Expected output:
- `runtime_info.extraction.model_name` shows `qwen2.5-1.5b-instruct` while `runtime_info.llm.model_name` can remain the general LLM.
- Chunk extraction returns a non-empty `summary`, useful string `tags`, and grounded typed `entities` for the default Personal KB sample.
- `document.tags` merge normalized chunk tags across repeated mentions.
- `document.entities` merge chunk entities by `(type, normalized_name)` and preserve `source_chunks`.
- Each chunk keeps its own `summary`, `tags`, `entities`, `tag_names`, `entity_names`, and embedding vector.
- The chunk inspection cell shows exactly what the LLM saw and what it returned.

Common failure points:
- LM Studio is not serving the OpenAI-compatible endpoint at the extraction base URL from `configs/default.yaml`.
- The `qwen2.5-1.5b-instruct` model is not loaded or available in LM Studio.
- The repo environment does not have the `openai` or `sentence-transformers` dependencies installed.
- The configured local embedding model is unavailable or returns vectors with the wrong dimension.


## Exercise and extension

Exercise:
- Replace `user_text` with a note that contains at least one date, one person, and one technology.
- Lower `chunk_size` to force more chunks, then compare the per-chunk prompts and the final document summary.

Optional extension:
- Change the source text so one entity appears across multiple chunks and inspect how the merged document tags and entities differ from the chunk-local outputs.


{'chunk_id': '97c9df52-77a3-58ee-a613-9c5b9959d6f5',
 'chunk_index': 0,
 'char_count': 2000,
 'source_ref': {'file_path': 'tests/fixtures/openwillis_speech_dataset_feature_reference.pdf',
  'page': 1,
  'section': None,
  'sheet': None,
  'cell_range': None},
 'text_preview': 'Mar 19, 2026 08:35 AM Speech characteristics v3.3 Speech characteristics v3.3 Dataset feature reference for openwillis-speech Project openwillis-speech Code ...'}

In [28]:
exercise_text = """Mar 19, 2026 08:35 AM Speech characteristics v3.3 Speech characteristics v3.3 Dataset feature reference for openwillis-speech Project openwillis-speech Code"""

parsed_exercise = build_parsed_document(exercise_text, "Exercise note")
exercise_document = build_document_record(parsed_exercise)
exercise_chunks = TxtChunker(chunk_size=120, chunk_overlap=20).chunk(
    parsed_exercise,
    document_id=exercise_document.document_id,
)
recording_client.history.clear()
try:
    exercise_enriched = extraction_service.enrich_chunks(exercise_chunks)
except ExtractionError as exc:
    failed_chunk = exercise_chunks[0] if exercise_chunks else None
    print_structured_output_failure(
        stage="Exercise chunk extraction",
        prompt=(build_chunk_extraction_prompt(failed_chunk) if failed_chunk else None),
        exc=exc,
        chunk_text=(failed_chunk.text if failed_chunk else None),
    )
    raise

exercise_embedded = embedding_service.embed_chunks(exercise_enriched)
try:
    exercise_aggregated = aggregation_service.aggregate_document(
        exercise_document,
        exercise_embedded,
    )
except ExtractionError as exc:
    print_structured_output_failure(
        stage="Exercise document aggregation",
        prompt=build_document_aggregation_prompt(exercise_document, exercise_embedded),
        exc=exc,
    )
    raise

print(json.dumps(build_final_payload(exercise_aggregated, exercise_embedded), indent=2))


{
  "document": {
    "document_id": "6b075b17619d4a3a",
    "source_id": "demo://custom-text",
    "source_type": "local_file",
    "file_path": "demo://custom-text",
    "file_name": "custom_text_demo.txt",
    "file_extension": "txt",
    "document_type": "text",
    "title": "Exercise note",
    "normalized_title": "exercise note",
    "summary": "The document discusses the reference of a new version of the speech characteristics dataset and mentions that it is part of an open willis speech project.",
    "tags": [
      {
        "tag_id": "tag::speech characteristics",
        "name": "speech characteristics",
        "normalized_name": "speech characteristics",
        "confidence": null,
        "source": "llm_extraction",
        "source_chunks": [
          "200b341e-c9c5-55a2-8255-623338d0e2d1"
        ],
        "created_at": null,
        "updated_at": null
      },
      {
        "tag_id": "tag::dataset",
        "name": "dataset",
        "normalized_name": "dataset",
 